# Ollama trên Google Colab cho StudyPlanner

Notebook chạy **Ollama** trên GPU miễn phí của Colab và mở API ra Internet bằng **Cloudflare Tunnel**
(không cần tài khoản). StudyPlanner gọi tới qua API chuẩn OpenAI.

**Cách dùng:** `Runtime → Change runtime type → T4 GPU`, rồi chạy lần lượt từng ô.
Ô cuối in ra dòng `AI_BASE_URL=...` để dán vào file `.env` của StudyPlanner (hoặc Environment trên Render).

**Lưu ý:**
- Phiên Colab miễn phí tự tắt khi không dùng (~90 phút) hoặc tối đa ~12 giờ. Mỗi lần chạy lại, **link tunnel đổi**, phải cập nhật `AI_BASE_URL`.
  Khi Colab tắt, trang Trợ lý AI tự chuyển sang phân tích dự phòng.
- Link tunnel **không có mật khẩu**: ai có link là dùng được model. Không chia sẻ link.
- Điều khoản Colab bản miễn phí hạn chế việc chạy dịch vụ web lâu dài; chỉ nên dùng để học tập / demo.

## 1. Cài Ollama

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Chạy máy chủ Ollama nền

In [ ]:
import os, subprocess, time, requests

env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_KEEP_ALIVE='30m')
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env,
                               stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        print(requests.get('http://localhost:11434/api/version', timeout=2).json())
        break
    except Exception:
        time.sleep(1)

## 3. Tải model

`qwen2.5:7b` viết tiếng Việt tốt và vừa GPU T4 (khoảng 5 GB). Có thể đổi sang `llama3.1:8b`, `gemma2:9b`...
Nhớ đặt `AI_MODEL` trong `.env` trùng tên model.

In [ ]:
MODEL = 'qwen2.5:7b'
!ollama pull {MODEL}

## 4. Mở API ra Internet bằng Cloudflare Tunnel

In [ ]:
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# --http-host-header: Ollama chỉ nhận Host là localhost
# Ghi log ra file (không dùng PIPE để tunnel không bị treo khi chạy lâu)
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--http-host-header', 'localhost:11434'],
    stdout=open('cloudflared.log', 'w'), stderr=subprocess.STDOUT,
)
public_url = None
for _ in range(60):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cloudflared.log').read())
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)
assert public_url, 'Chưa lấy được link tunnel, xem cloudflared.log'

print('Dán các dòng sau vào .env của StudyPlanner:\n')
print(f'AI_BASE_URL={public_url}/v1')
print(f'AI_MODEL={MODEL}')

## 5. Thử gọi như StudyPlanner (API chuẩn OpenAI)

In [ ]:
!pip -q install openai
from openai import OpenAI

client = OpenAI(base_url=f'{public_url}/v1', api_key='ollama', timeout=120)
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Trả về JSON {"loi_chao": "..."} chào sinh viên bằng tiếng Việt.'}],
    response_format={'type': 'json_object'},
)
print(reply.choices[0].message.content)

## 6. Giữ phiên chạy

Để tab Colab mở trong lúc demo. Ô dưới in trạng thái mỗi 5 phút; dừng ô (■) khi xong.

In [ ]:
while True:
    alive = ollama_proc.poll() is None and tunnel_proc.poll() is None
    print(time.strftime('%H:%M:%S'), 'Ollama + tunnel đang chạy' if alive else 'Đã dừng, hãy chạy lại các ô trên')
    time.sleep(300)